## **Gradient Boosting**

Gradient Boosting ek powerful ensemble learning technique hai jo weak learners (usually small decision trees) ko combine karke ek strong predictive model banata hai. Iska main idea ye hai ki har naya model pichle model ki **galatiyon (errors)** ko sudhaarne ki koshish karta hai.

---

## 1. Simple Intuition (Hinglish mein)

Maano aapko ek target value $y$ predict karni hai:

1. Sabse pehle, aap ek bahut simple model banate ho (jaise ki average value). Isse hum $F_0$ kehte hain.
2. Ab, is simple model mein kuch error hoga. Is error ko hum **Residual** kehte hain ($y - F_0$).
3. Next step mein, aap ek naya model ($h_1$) banate ho jo target $y$ ko predict nahi karta, balki us **Residual (error)** ko predict karta hai.
4. Aap apne purane model mein is naye model ka output add kar dete ho: $F_1 = F_0 + \eta h_1$. Yahan $\eta$ (learning rate) hai jo model ko overfit hone se bachata hai.
5. Ye process chalti rehti hai jab tak error minimze na ho jaye.

**Moral of the story:** Gradient Boosting "Residuals" ke peeche bhagta hai taaki error zero ho sake.

---

## 2. Mathematical Intuition (In Detail)

Gradient Boosting ko mathematical perspective se **Gradient Descent** ki tarah dekha ja sakta hai, jahan hum Loss Function ko minimize karte hain.

### Step 1: Initialize the Model

Sabse pehle hum model ko ek constant value se start karte hain:


$$F_0(x) = \underset{\gamma}{\arg\min} \sum_{i=1}^{n} L(y_i, \gamma)$$


Agar hum Mean Squared Error (MSE) use kar rahe hain, to $F_0$ simply target values ka **mean** hoga.

### Step 2: Iterative Training (for $m = 1$ to $M$)

Har iteration $m$ ke liye, hum ye steps follow karte hain:

**A. Calculate Pseudo-Residuals:**
Hum dekhte hain ki model ka prediction aur actual value mein kitna "gradient" (slope) hai. Ye residuals loss function ka negative gradient hote hain:


$$r_{im} = -\left[ \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \right]_{F(x)=F_{m-1}(x)}$$


*Intuition:* Ye batata hai ki prediction ko kis direction mein badalna hai taaki loss kam ho.

**B. Fit a Weak Learner:**
Hum ek decision tree ($h_m(x)$) train karte hain jo in residuals $r_{im}$ ko predict kare.

**C. Compute Step Size ($\gamma_m$):**
Hum determine karte hain ki is naye tree ki kitni "weightage" honi chahiye taaki overall loss minimize ho:


$$\gamma_m = \underset{\gamma}{\arg\min} \sum_{i=1}^{n} L(y_i, F_{m-1}(x_i) + \gamma h_m(x_i))$$

**D. Update the Model:**
Naye tree ko purane model mein add kar diya jata hai:


$$F_m(x) = F_{m-1}(x) + \nu \cdot \gamma_m h_m(x)$$


Yahan $\nu$ (nu) **Learning Rate** hai.

---

## 3. Key Components Summary

| Component | Description |
| --- | --- |
| **Loss Function** | Isse hum measure karte hain ki model kitna galat hai (e.g., MSE for regression, Log-loss for classification). |
| **Weak Learner** | Usually small Decision Trees (Regression Trees) jinka kaam residuals ko fit karna hota hai. |
| **Additive Model** | Hum purane trees ko change nahi karte, bas naye trees add karte jaate hain. |



Bilkul! Gradient Boosting ko dummy data (7 rows) par step-by-step apply karte hain. Hum **Regression** ka example lenge jahan hume **Salary** predict karni hai based on **Experience**.

### Dummy Dataset

Hamara goal hai `Salary` predict karna.

| ID | Experience (Years) | Salary (Target $y$) |
| --- | --- | --- |
| 1 | 2 | 40k |
| 2 | 5 | 70k |
| 3 | 1 | 30k |
| 4 | 8 | 100k |
| 5 | 3 | 50k |
| 6 | 6 | 80k |
| 7 | 4 | 60k |

---

### Step 1: Initial Prediction ($F_0$)

Gradient Boosting hamesha ek constant value se start hota hai. Regression ke liye hum target ka **Mean** lete hain.

$$F_0 = \frac{40+70+30+100+50+80+60}{7} = \mathbf{61.4k}$$

Ab hamara initial model sabke liye **61.4k** predict kar raha hai.

---

### Step 2: Calculate Residuals ($r_{i1}$)

Residual matlab error: $Residual = Actual - Predicted$.

| ID | Actual ($y$) | $F_0$ (Pred) | **Residual ($r_1$)** |
| --- | --- | --- | --- |
| 1 | 40 | 61.4 | **-21.4** |
| 2 | 70 | 61.4 | **8.6** |
| 3 | 30 | 61.4 | **-31.4** |
| 4 | 100 | 61.4 | **38.6** |
| 5 | 50 | 61.4 | **-11.4** |
| 6 | 80 | 61.4 | **18.6** |
| 7 | 60 | 61.4 | **-1.4** |

---

### Step 3: Train a Decision Tree on Residuals

Ab hum ek Decision Tree (Weak Learner $h_1$) banayenge. Par dhyan rahe, ye tree `Salary` predict nahi karega, balki upar waale **Residuals ($r_1$)** ko predict karega.

Maano hamara tree "Experience" ke basis par residuals ko split karta hai:

* **Node 1 (Exp < 3.5):** Average Residual = $\frac{-21.4 - 31.4 - 11.4}{3} = \mathbf{-21.4}$
* **Node 2 (Exp > 3.5):** Average Residual = $\frac{8.6 + 38.6 + 18.6 - 1.4}{4} = \mathbf{16.1}$

---

### Step 4: Update the Model ($F_1$)

Ab hum apne model ko update karenge. Hum directly pura residual add nahi karte, kyunki isse **overfitting** ho sakti hai. Hum ek **Learning Rate ($\eta$)** use karte hain (Maano $\eta = 0.1$).

Formula: $F_1 = F_0 + \eta \cdot h_1$

* **For ID 1 (Exp=2):** New Pred = $61.4 + (0.1 \times -21.4) = 59.26$
* **For ID 4 (Exp=8):** New Pred = $61.4 + (0.1 \times 16.1) = 63.01$

Ab aap dekh sakte hain ki predictions actual value ki taraf dheere-dheere move kar rahi hain (e.g., ID 4 ka prediction 61.4 se 63.01 ho gaya, jo 100 ke thoda aur paas hai).

---

## Step 5: Iteration Repeat (Iteration 2)

* **Learning Rate ($\eta$):** 0.1
* **Tree 1 Split:** Experience < 3.5 (IDs 1, 3, 5) aur Experience > 3.5 (IDs 2, 4, 6, 7).

---

### Final Predictions ($F_1$) and New Residuals ($r_2$)

Yahan calculation ka formula hai: $F_1 = F_0 + (\eta \times \text{Tree}_1 \text{ Prediction})$.
Yaad rahe $F_0 = 61.4$.

| ID | Exp | Salary ($y$) | Tree 1 Pred (Avg Residual) | New Prediction ($F_1$) | **New Residual ($r_2 = y - F_1$)** |
| --- | --- | --- | --- | --- | --- |
| **1** | 2 | 40 | -21.4 | $61.4 + (0.1 \times -21.4) = \mathbf{59.26}$ | **-19.26** |
| **2** | 5 | 70 | 16.1 | $61.4 + (0.1 \times 16.1) = \mathbf{63.01}$ | **6.99** |
| **3** | 1 | 30 | -21.4 | $61.4 + (0.1 \times -21.4) = \mathbf{59.26}$ | **-29.26** |
| **4** | 8 | 100 | 16.1 | $61.4 + (0.1 \times 16.1) = \mathbf{63.01}$ | **36.99** |
| **5** | 3 | 50 | -21.4 | $61.4 + (0.1 \times -21.4) = \mathbf{59.26}$ | **-9.26** |
| **6** | 6 | 80 | 16.1 | $61.4 + (0.1 \times 16.1) = \mathbf{63.01}$ | **16.99** |
| **7** | 4 | 60 | 16.1 | $61.4 + (0.1 \times 16.1) = \mathbf{63.01}$ | **-3.01** |

---

### Observations (Kya badla?)

1. **Directional Improvement:** * ID 4 (Salary 100) ka prediction 61.4 se badhkar **63.01** hua. (Sahi direction mein gaya!)
* ID 3 (Salary 30) ka prediction 61.4 se ghatkar **59.26** hua. (Ye bhi sahi direction mein hai!)


2. **Residual Reduction:** Agar aap purane residuals ($r_1$) aur naye residuals ($r_2$) ko compare karein, to errors dheere-dheere kam ho rahe hain.
3. **Next Step:** Ab **Iteration 2** mein hum ek naya Tree ($h_2$) banayenge jo specifically in **$r_2$** values (last column) ko predict karne ki koshish karega.

---

### What's next?

Ye process aise hi 100 ya 500 baar (n_estimators) chalti rahegi jab tak error minimum na ho jaye.

 
### Key Summary

1. **Base Model:** Sabse pehle Mean nikala.
2. **Mistakes:** Dekha ki Mean se kitni galati ho rahi hai (Residuals).
3. **Correction:** Ek naya chota tree banaya jo sirf un galatiyon ko sudhaarne ki koshish kare.
4. **Learning Rate:** Sudhaar ko thoda-thoda karke apply kiya taaki model "ratta" (overfit) na mare.
